# 📖 RAG 101 — Guided Workbook

**Author:** Gian-Carlo J  **Date:** June 19 2026

---

## What You Are Building

```
[Document / Web Page]
        │
        ▼  Exercise 2 — Load
  list[Document]
        │
        ▼  Exercise 3 — Chunk
  list[Chunk]  (512 chars, 64 overlap)
        │
        ▼  Exercise 4 — Embed + Index
  ChromaDB Vector Store
        │
  User Query ──► Retriever (top-4 chunks)
                        │
        ▼  Exercise 5 — RAG Chain
  Prompt Template  +  gpt-4o-mini
                        │
                    Answer
                        │
        ▼  Exercises 7–9 — Evaluate
  RAGAS Scores  ──►  CSV Export
```

**Stack:** LangChain · ChromaDB · OpenAI · RAGAS · HuggingFace Datasets

**Prerequisite:** An OpenAI API key — get one at https://platform.openai.com


---

## 🗂️ Git Setup

Run the cell below **once** before you start. It wires up your local folder to the
GitHub repo `Dashakid/RAG-101` and creates a `.gitignore` that protects your API key.

> 🔒 **Rule:** Never let `.env` appear in `git status` as a staged or tracked file.
> The setup cell and every checkpoint cell check this automatically.


In [1]:
# ════════════════════════════════════════════════════════════════
# GIT SETUP — Run once, then never again
# ════════════════════════════════════════════════════════════════
import subprocess, os
from pathlib import Path

def run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.stdout.strip(): print(r.stdout.strip())
    if r.stderr.strip(): print(r.stderr.strip())

# Confirm .env is in .gitignore
gi = Path(".gitignore").read_text() if Path(".gitignore").exists() else ""
assert ".env" in gi, "ERROR: .env is not in .gitignore — check the file before proceeding"
print("✓ .env is protected by .gitignore")

# Show remote
remote = subprocess.run("git remote get-url origin", shell=True,
                        capture_output=True, text=True).stdout.strip()
print(f"✓ Remote: {remote}")

# Show current branch
branch = subprocess.run("git branch --show-current", shell=True,
                        capture_output=True, text=True).stdout.strip()
print(f"✓ Branch: {branch}")
print("\nGit setup confirmed. Ready to start exercises.")


✓ .env is protected by .gitignore
✓ Remote: git@github.com:Dashakid/RAG-101.git
✓ Branch: main

Git setup confirmed. Ready to start exercises.


---

## Exercise 0 — Install Dependencies

**Instructions:**
1. Just run this cell — nothing to write yet
2. Read the package list and look up any package you don't recognise
3. The `-U` flag upgrades to latest versions; `-q` keeps output clean

**Expected output:** `✓ Packages ready.`


In [2]:
# ════════════════════════════════════════════════════════════════
# EXERCISE 0 — Install & Upgrade All Dependencies
# ════════════════════════════════════════════════════════════════

%pip install -U -q pip

%pip install -q \
    "langchain>=0.3" \
    "langchain-community>=0.3" \
    "langchain-chroma>=0.1" \
    "langchain-text-splitters>=0.2" \
    "langchain-google-genai>=4.0" \
    "chromadb>=0.5,<0.6" \
    "ragas>=0.2" \
    "datasets>=2.19" \
    "pandas>=2.0" \
    "matplotlib>=3.8" \
    "seaborn>=0.13" \
    "python-dotenv>=1.0" \
    "beautifulsoup4>=4.12"

print("✓ Packages ready. Move on to Exercise 1.")

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
✓ Packages ready. Move on to Exercise 1.


---

## Exercise 1 — Imports & API Key

### Concept: why we import at the top
Python needs to know which external libraries you're using before you call them.
Grouping all imports at the top of the notebook is a best practice — it makes
dependencies visible and prevents "name not defined" errors halfway through.

### What to import

| What you need | Package path | Class / function |
|--------------|-------------|-----------------|
| Load web pages | `langchain_community.document_loaders` | `WebBaseLoader` |
| Split text | `langchain.text_splitter` | `RecursiveCharacterTextSplitter` |
| Create embeddings | `langchain_openai` | `OpenAIEmbeddings` |
| Chat LLM | `langchain_openai` | `ChatOpenAI` |
| Local vector DB | `langchain_chroma` | `Chroma` |
| Prompt template | `langchain_core.prompts` | `ChatPromptTemplate` |
| Parse LLM output | `langchain_core.output_parsers` | `StrOutputParser` |
| Pass data through | `langchain_core.runnables` | `RunnablePassthrough` |
| RAGAS runner | `ragas` | `evaluate` |
| RAGAS metrics | `ragas.metrics` | `faithfulness`, `answer_relevancy`, `context_recall`, `context_precision` |
| Dataset format | `datasets` | `Dataset` |
| Data analysis | `pandas` | import as `pd` |

### API Key

Create a `.env` file in this folder containing one line:
```
OPENAI_API_KEY=sk-your-key-here
```
Then call `load_dotenv()` to read it. Import `load_dotenv` from `dotenv`.

**Instructions:**
1. Write one `from ... import ...` line for each row in the table above
2. Call `load_dotenv()` to load your `.env` file
3. Read `OPENAI_API_KEY` from the environment with `os.getenv(...)`
4. Run the self-check — it will tell you exactly which imports are missing

**Expected output:** `✓ All imports found!` and `✓ API key loaded`


In [3]:
# Compatibility patch: stub VertexAI shim removed in langchain-community 0.3+
# RAGAS imports it internally; this notebook uses Gemini so the stub is never called.
import sys
if "langchain_community.chat_models.vertexai" not in sys.modules:
    from types import ModuleType
    _stub = ModuleType("langchain_community.chat_models.vertexai")
    _stub.ChatVertexAI = None
    sys.modules["langchain_community.chat_models.vertexai"] = _stub

# ════════════════════════════════════════════════════════════════
# EXERCISE 1 — Imports & API Key Configuration
# ════════════════════════════════════════════════════════════════

import os
import warnings
warnings.filterwarnings("ignore")

# 1. Import load_dotenv
from dotenv import load_dotenv

# 2. Import the LangChain classes
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# 3. Import RAGAS — using 0.2.x API (metric classes + EvaluationDataset)
from ragas import evaluate
from ragas.dataset_schema import EvaluationDataset, SingleTurnSample
from ragas.metrics import Faithfulness, AnswerRelevancy, ContextRecall, ContextPrecision

# 4. Import pandas
import pandas as pd

# 5. Import Gemini embeddings and chat model  ← FIX: single underscore
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI

# ── SELF CHECK ───────────────────────────────────────────────
try:
    _ = [WebBaseLoader, RecursiveCharacterTextSplitter,
         GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI, Chroma,
         ChatPromptTemplate, StrOutputParser, RunnablePassthrough,
         evaluate, EvaluationDataset, SingleTurnSample,
         Faithfulness, AnswerRelevancy, ContextRecall, ContextPrecision,
         pd]
    print("✓ All imports found!")
except NameError as e:
    print(f"✗ Missing: {e}  — add the import above and re-run")

# 6. Call load_dotenv()
load_dotenv()

# 7. Store your API key  ← FIX: GOOGLE_API_KEY not OPENAI_API_KEY
api_key = os.getenv("GOOGLE_API_KEY")

# ── SELF CHECK ───────────────────────────────────────────────
try:
    if api_key and len(api_key) > 10 and "your" not in api_key.lower():  # ← FIX: no sk- check
        print(f"✓ API key loaded  ({api_key[:8]}...{api_key[-4:]})")
    else:
        print("✗ API key missing or looks like a placeholder — check your .env file")
except NameError:
    print("✗ api_key not defined — did you write step 7 above?")

USER_AGENT environment variable not set, consider setting it to identify your requests.


ModuleNotFoundError: No module named 'langchain_community.chat_models.vertexai'

---

## Exercise 2 — Load a Document

### Concept: Document Loaders

LangChain wraps many data sources behind a single interface. Every loader exposes
one method: `.load()` which returns a `list[Document]`.

A `Document` object has two attributes:
```python
doc.page_content   # the raw text string
doc.metadata       # dict — source URL, page number, filename, etc.
```

We are loading the Wikipedia article on *Retrieval-Augmented Generation* — meta
and perfect for demonstrating what RAG can do.

**URL:** `https://en.wikipedia.org/wiki/Retrieval-augmented_generation`

### Loader options (for your future projects)

| Your data source | Loader class |
|-----------------|-------------|
| Single PDF | `PyPDFLoader("file.pdf")` |
| Folder of PDFs | `DirectoryLoader("./data", glob="**/*.pdf")` |
| Plain text file | `TextLoader("file.txt")` |
| Website / blog | `WebBaseLoader("https://...")` |
| Notion database | `NotionDBLoader(...)` |

**Instructions:**
1. Create a `WebBaseLoader` instance — pass `SOURCE_URL` as the argument
2. Call `.load()` and store the result in `documents`
3. Print: number of documents, total character count, first 500 chars of `documents[0].page_content`

**Expected output:** 1 document, several thousand characters of Wikipedia text.


In [ ]:
# ════════════════════════════════════════════════════════════════
# EXERCISE 2 — Load the Source Document
# ════════════════════════════════════════════════════════════════

SOURCE_URL = "https://en.wikipedia.org/wiki/Retrieval-augmented_generation"

# ── YOUR CODE HERE ────────────────────────────────────────────
# Step 1: Create a WebBaseLoader — pass SOURCE_URL as the argument
loader = WebBaseLoader(SOURCE_URL)

# Step 2: Call loader.load() — store result in `documents`
documents = loader.load()

# Step 3: Print stats
print(f"Documents loaded : {len(documents)}")
print(f"Total characters : {sum(len(d.page_content) for d in documents):,}")
print(f"\nFirst 500 chars:\n{documents[0].page_content[:500]}")


# ── SELF CHECK ───────────────────────────────────────────────
assert documents not in (None, []), "✗ documents is empty — did you call loader.load()?"
assert len(documents) > 0,          "✗ No documents returned — check the URL"
assert len(documents[0].page_content) > 500, \
    "✗ Document is too short — something went wrong with loading"
print(f"✓ Document loaded  ({len(documents[0].page_content):,} characters)")


---

## Exercise 3 — Chunk the Document

### Concept: Why We Chunk

An LLM has a fixed context window (e.g. 128k tokens for GPT-4o). More importantly,
embedding a 50,000-character article as a single vector produces a vague "average"
of everything in it — too blurry for precise retrieval.

Chunking breaks the document into short, focused passages so each embedding
represents one specific idea. Overlapping chunks ensures nothing is cut off mid-thought.

```
Chunk 1: [═══════════════════════════════]
Chunk 2:                    [═══════════════════════════════]
                  ◄── overlap 64 chars ──►
```

### RecursiveCharacterTextSplitter

Splits on natural boundaries in this priority order:
`\n\n` → `\n` → `. ` → ` ` → individual characters

This preserves semantic coherence far better than slicing at a fixed character count.

**Parameters to use:**

| Parameter | Value | Why |
|-----------|-------|-----|
| `chunk_size` | `512` | ~2–3 sentences — specific enough for precise retrieval |
| `chunk_overlap` | `64` | ~12% — prevents boundary cut-offs |
| `length_function` | `len` | measure in characters (not tokens) |

**Instructions:**
1. Create a `RecursiveCharacterTextSplitter` with the values in the table above
2. Call `.split_documents(documents)` — store result in `chunks`
3. Print: total chunks, average chunk length, content of the first chunk

**Expected output:** 50–150 chunks depending on the article length.


In [ ]:
# ════════════════════════════════════════════════════════════════
# EXERCISE 3 — Chunk the Document
# ════════════════════════════════════════════════════════════════

# Step 1: Create the text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=64,
    length_function=len,
)

# Step 2: Split the documents
chunks = text_splitter.split_documents(documents)

# Step 3: Print stats
avg = sum(len(c.page_content) for c in chunks) / len(chunks)
print(f"Total chunks        : {len(chunks)}")
print(f"Average chunk length: {avg:.0f} chars")
print(f"\nFirst chunk preview:\n{chunks[0].page_content}")


# ── SELF CHECK ───────────────────────────────────────────────
assert chunks not in (None, []),      "✗ chunks is empty — did you call split_documents()?"
assert len(chunks) > 10,              f"✗ Only {len(chunks)} chunks — check chunk_size parameter"
assert all(len(c.page_content) <= 600 for c in chunks), \
    "✗ Some chunks exceed 600 chars — double-check chunk_size=512"
avg = sum(len(c.page_content) for c in chunks) / len(chunks)
print(f"✓ {len(chunks)} chunks created  (avg {avg:.0f} chars each)")

---

## 📌 Git Checkpoint 1 — Document Loaded & Chunked

**Commit before moving on.**  
Chunking strategy is a key design decision — committing it here lets you
experiment with different `chunk_size` values later and diff the results.

**Fill in the `NOTES` string below, then run the cell.**


In [ ]:
# ════════════════════════════════════════════════════════════════
# GIT CHECKPOINT 1
# ════════════════════════════════════════════════════════════════
# Edit NOTES with YOUR observations, then run.

NOTES = """
Exercises: 1 (imports), 2 (document loading), 3 (chunking)
Status: ✓ / ⚠ / ✗   ← delete two

Chunk count     : ???   ← fill in len(chunks)
Avg chunk length: ???   ← fill in average chars
Chunk size used : 512   overlap: 64

Notes (what you learned / what was confusing):
-
-
"""

import subprocess

# Safety: block commit if .env is staged
staged = subprocess.run("git diff --cached --name-only",
                        shell=True, capture_output=True, text=True).stdout
assert ".env" not in staged, "🚨  .env is staged! Run: git reset HEAD .env  then retry"
print("✓ .env safety check passed")

subprocess.run("git add rag_eval_workflow.ipynb", shell=True)
subprocess.run(["git", "commit", "-m",
    f"[RAG-1] Document loaded and chunked\n\n{NOTES.strip()}"])
print("\n✓ Committed! Run `make check-git` to verify.")
print("  Push when ready:  git push origin main")


---

## Exercise 4 — Embeddings & Vector Store

### Concept: What Is an Embedding?

An embedding model converts any string of text into a fixed-length list of numbers
(a vector). Texts that mean similar things produce vectors that are close together
in that numeric space.

```
"What is RAG?"    →  [0.021, -0.134, 0.872, ..., 0.003]   ← 1536 numbers
"RAG overview"    →  [0.019, -0.130, 0.869, ..., 0.001]   ← very close!
"Chocolate cake"  →  [-0.412, 0.823, -0.011, ..., 0.234]  ← far away
```

Retrieval works by embedding the user's question and finding the chunk vectors
nearest to it — those are the most relevant passages.

### Embedding model

```python
OpenAIEmbeddings(model="text-embedding-3-small")
```
1536 dimensions, fast, cheap (~$0.02 per 1M tokens).

### ChromaDB

Runs **locally** — no cloud account, no network latency. Stores your vectors and
runs similarity search in milliseconds.

```python
Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="rag_wiki",
)
```

After building the store, wrap it in a **retriever**:
```python
vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})
```
`k=4` means return the 4 most relevant chunks per query.

**Instructions:**
1. Initialize `OpenAIEmbeddings` with `model="text-embedding-3-small"`
2. Call `Chroma.from_documents(...)` with your chunks and embeddings — store as `vector_store`
3. Create a retriever with `search_type="similarity"` and `k=4`
4. Test it: call `retriever.invoke("What problem does RAG solve?")` and print each chunk

> ⏱ Step 2 makes API calls to embed all chunks — takes ~10–30 seconds.

**Expected output:** 4 chunks returned, each showing relevant text about RAG.


In [ ]:
# ════════════════════════════════════════════════════════════════
# EXERCISE 4 — Embeddings & Vector Store
# ════════════════════════════════════════════════════════════════

# Safety: reload .env in case kernel was restarted without re-running Exercise 1
import os
from dotenv import load_dotenv
load_dotenv()

_key = os.getenv("OPENAI_API_KEY", "")
assert _key and _key.startswith("sk-") and "your" not in _key.lower(), (
    "✗ OPENAI_API_KEY is not set.\n"
    "  1. Create a .env file in this folder: OPENAI_API_KEY=sk-...\n"
    "  2. Re-run Exercise 1, then retry this cell."
)
print(f"✓ API key confirmed ({_key[:8]}...{_key[-4:]})")

# Step 1: Initialize the embedding model
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Step 2: Build the Chroma vector store from your chunks
print("Embedding chunks — this takes ~10-30 seconds...")
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="rag_wiki",
)

# Step 3: Create a retriever
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4},
)

# Step 4: Test the retriever
TEST_QUERY = "What problem does RAG solve?"
results = retriever.invoke(TEST_QUERY)
for i, doc in enumerate(results, 1):
    print(f"  Chunk {i}: {doc.page_content[:150].strip()}...")


# ── SELF CHECK ───────────────────────────────────────────────
assert embeddings  is not None, "✗ embeddings not set"
assert vector_store is not None, "✗ vector_store not built — did you call Chroma.from_documents()?"
assert retriever   is not None, "✗ retriever not created — did you call .as_retriever()?"

test_results = retriever.invoke("What is retrieval augmented generation?")
assert len(test_results) == 4, \
    f"✗ Expected 4 results, got {len(test_results)} — check search_kwargs={{'k': 4}}"
assert all(hasattr(r, "page_content") for r in test_results), \
    "✗ Results don't look like Document objects — check Chroma.from_documents arguments"
print(f"\n✓ Vector store ready — {vector_store._collection.count()} vectors indexed")
print(f"✓ Retriever returns {len(test_results)} chunks per query")


---

## Exercise 5 — Build the RAG Chain

### Concept: LangChain Expression Language (LCEL)

LCEL lets you chain components together with the `|` pipe operator.
Each component receives the output of the previous one.

The full data flow:

```
User Question (string)
   │
   ├─────────────────────────────────────────────┐
   ▼                                             ▼
retriever.invoke(question)           RunnablePassthrough()
   │                                             │
   ▼                                             │
[Doc1, Doc2, Doc3, Doc4]                         │
   │                                             │
   ▼                                             │
format_docs(docs) → single string                │
   │                                             │
   └──────────── ChatPromptTemplate ◄────────────┘
                        │
                  ChatOpenAI LLM
                        │
                  StrOutputParser
                        │
                   Final Answer (string)
```

### Your system prompt must:
- Tell the LLM to answer **only from the provided context**
- Include a `{context}` placeholder (filled by `format_docs`)
- Give a clear fallback: *"If the answer is not in the context, say 'I don't know.'"*

### Assembling the chain with LCEL

```python
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)
```

**Instructions:**
1. Write `format_docs(docs)` — joins each doc's `.page_content` with `"\n\n"` as separator
2. Write your `SYSTEM_TEMPLATE` string — must contain `{context}`
3. Build `prompt` with `ChatPromptTemplate.from_messages([("system", ...), ("human", "{question}")])`
4. Initialize `llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)`
5. Assemble `rag_chain` using the LCEL pipe syntax above
6. Test it: run `rag_chain.invoke("What is retrieval-augmented generation?")`

**Expected output:** A clear, grounded answer about RAG based on the Wikipedia article.


In [ ]:
# ════════════════════════════════════════════════════════════════
# EXERCISE 5 — Assemble the RAG Chain
# ════════════════════════════════════════════════════════════════

# Step 1: format_docs helper — joins page_content with double newlines
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


# Step 2: System prompt template
SYSTEM_TEMPLATE = """You are a helpful assistant for question-answering tasks.
Use ONLY the following retrieved context to answer the question.
If the answer is not in the context, say "I don't know."
Do not add any information that is not explicitly in the context.

Context:
{context}"""


# Step 3: Build the prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_TEMPLATE),
    ("human", "{question}"),
])


# Step 4: Initialize the LLM — temperature=0 for deterministic answers
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


# Step 5: Assemble the chain with LCEL
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)


# Step 6: Test it
test_answer = rag_chain.invoke("What is retrieval-augmented generation?")
print(test_answer)


# ── SELF CHECK ───────────────────────────────────────────────
sample_docs = retriever.invoke("what is RAG?")
formatted = format_docs(sample_docs)
assert isinstance(formatted, str) and len(formatted) > 100, \
    "✗ format_docs should return a non-empty string joined from doc.page_content"
print("✓ format_docs works")

assert "{context}" in SYSTEM_TEMPLATE, \
    "✗ SYSTEM_TEMPLATE must contain {context} — the retrieved chunks go here"
assert prompt    is not None, "✗ prompt not built — call ChatPromptTemplate.from_messages(...)"
assert llm       is not None, "✗ llm not initialized — call ChatOpenAI(...)"
assert rag_chain is not None, "✗ rag_chain not assembled — use the LCEL pipe syntax"

print("Running live chain test...")
answer = rag_chain.invoke("What is retrieval-augmented generation?")
assert isinstance(answer, str) and len(answer) > 20, \
    "✗ Chain returned empty or unexpected output"
print(f"✓ Chain works!\n\nTest answer:\n{answer}")


---

## 📌 Git Checkpoint 2 — Core Pipeline Working

**The hardest part is done — commit it.**  
A working chain is your v1 baseline. Every improvement after this point
can be measured against it.

**Fill in `NOTES` with one of your test answers, then run the cell.**


In [ ]:
# ════════════════════════════════════════════════════════════════
# GIT CHECKPOINT 2
# ════════════════════════════════════════════════════════════════

NOTES = """
Exercises: 4 (embeddings + vector store), 5 (RAG chain)
Status: ✓ / ⚠ / ✗   ← delete two

Embedding model : text-embedding-3-small
LLM             : gpt-4o-mini  temperature=0
Chunks indexed  : ???   ← fill in vector_store._collection.count()

Sample test answer:
  Q: What is RAG?
  A: ???   ← paste your answer here

Notes:
-
-
"""

import subprocess
staged = subprocess.run("git diff --cached --name-only",
                        shell=True, capture_output=True, text=True).stdout
assert ".env" not in staged, "🚨  .env is staged! Run: git reset HEAD .env"
print("✓ .env safety check passed")

subprocess.run("git add rag_eval_workflow.ipynb", shell=True)
subprocess.run(["git", "commit", "-m",
    f"[RAG-2] Vector store + RAG chain working\n\n{NOTES.strip()}"])
print("\n✓ Committed!  Push when ready: git push origin main")


---

## Exercise 6 — Test Your Pipeline with Real Queries

### Concept: Manual Testing Before Formal Evaluation

Before running automated metrics, probe the pipeline yourself. You are looking for:

| Test type | What to ask | What to look for |
|-----------|------------|-----------------|
| **In-scope fact** | Something clearly in the article | A specific, grounded answer |
| **Conceptual** | "What is the difference between X and Y?" | Multi-sentence explanation |
| **Out-of-scope** | Something NOT in the article | Should say "I don't know" |
| **Edge case** | Vague or ambiguous question | How does it handle uncertainty? |

**Instructions:**
1. Write at least **6 questions** in `my_questions` — cover all 4 test types above
2. Loop through them: call `rag_chain.invoke(q)` and print each answer
3. Observe: does the model ever hallucinate? Does it correctly say "I don't know"?

**Expected output:** 6+ question/answer pairs printed clearly.


In [ ]:
# ════════════════════════════════════════════════════════════════
# EXERCISE 6 — Interactive Testing
# ════════════════════════════════════════════════════════════════

my_questions = [
    "What is retrieval-augmented generation?",                    # in-scope: simple fact
    "What problem does RAG solve for large language models?",     # in-scope: conceptual
    "How does RAG differ from fine-tuning a language model?",     # in-scope: comparison
    "What is the boiling point of water on Mars?",                # out-of-scope
    "What types of external knowledge sources can RAG use?",      # in-scope: your choice
    "Who introduced RAG and in what year?",                       # in-scope: simple fact
    "What",                                                       # edge case: vague/ambiguous
]

for q in my_questions:
    answer = rag_chain.invoke(q)
    print(f"Q: {q}")
    print(f"A: {answer}")
    print("-" * 60)


# ── SELF CHECK ───────────────────────────────────────────────
assert len(my_questions) >= 6, \
    f"✗ Write at least 6 questions — you have {len(my_questions)}"
print(f"✓ Tested {len(my_questions)} questions.")
print("\nReflection — answer these before moving on:")
print("  1. Did the model ever include information NOT in the article?")
print("  2. Did it correctly refuse off-topic questions?")
print("  3. Which answer impressed you most?")


---

## 📌 Git Checkpoint 3 — Manual Testing Complete

**What to note:** Your test questions reveal your intuition about the system.
Writing them down before formal evaluation is valuable for your portfolio.


In [ ]:
# ════════════════════════════════════════════════════════════════
# GIT CHECKPOINT 3
# ════════════════════════════════════════════════════════════════

NOTES = f"""
Exercises: 6 (interactive testing)
Status: ✓ / ⚠ / ✗   ← delete two

Questions tested   : {len(my_questions)}
Hallucination seen : yes / no
"I don't know" worked: yes / no

Most interesting answer:
  Q: ???
  A: ???

Notes:
-
-
"""

import subprocess
staged = subprocess.run("git diff --cached --name-only",
                        shell=True, capture_output=True, text=True).stdout
assert ".env" not in staged, "🚨  .env is staged! Run: git reset HEAD .env"
print("✓ .env safety check passed")

subprocess.run("git add rag_eval_workflow.ipynb", shell=True)
subprocess.run(["git", "commit", "-m",
    f"[RAG-3] Manual query testing — {len(my_questions)} questions\n\n{NOTES.strip()}"])
print("✓ Committed!  Push when ready: git push origin main")


---

## Exercise 7 — Build a Golden Evaluation Dataset

### Concept: What Is a Golden Dataset?

A golden dataset is your **ground truth** — questions with known correct answers
that *you* write before seeing the model's output. It is the foundation of
objective evaluation.

Each row has 4 fields:

| Field | Type | Description |
|-------|------|-------------|
| `question` | `str` | A realistic user question |
| `answer` | `str` | The answer your RAG chain generated |
| `contexts` | `list[str]` | Raw chunk text retrieved (not the Document objects) |
| `ground_truth` | `str` | Your handwritten reference answer |

### Rules for writing good questions

- Write your `ground_truths` **before** running the model — don't peek
- Cover different sections of the article (not all from the introduction)
- Include simple lookups, conceptual questions, and at least one multi-hop question
- Write **at least 10** — fewer gives statistically unreliable metric averages

### Capturing contexts correctly

RAGAS needs the **raw text strings** from the retrieved chunks, not Document objects:
```python
docs = retriever.invoke(question)
context_texts = [doc.page_content for doc in docs]   # list[str]
```

### Building the Dataset

```python
eval_dataset = Dataset.from_dict({
    "question":    questions,           # list[str]
    "answer":      generated_answers,   # list[str]
    "contexts":    retrieved_contexts,  # list[list[str]]
    "ground_truth": ground_truths,      # list[str]
})
```

**Instructions:**
1. Write 10 questions in `questions` and matching reference answers in `ground_truths`
2. Loop through questions: retrieve chunks, generate answers, append to `generated_answers` and `retrieved_contexts`
3. Build `eval_dataset` with `Dataset.from_dict(...)`

**Expected output:** `Dataset({features: ['question','answer','contexts','ground_truth'], num_rows: 10})`


In [ ]:
# ════════════════════════════════════════════════════════════════
# EXERCISE 7 — Build the Golden Evaluation Dataset
# ════════════════════════════════════════════════════════════════

# Part A: Questions and ground-truth reference answers
questions = [
    "What is retrieval-augmented generation?",
    "What problem does RAG solve compared to standard LLMs?",
    "Who introduced the concept of RAG and when?",
    "What are the main components of a RAG pipeline?",
    "How does RAG differ from fine-tuning a language model?",
    "What types of knowledge sources can RAG retrieve from?",
    "What is the role of the retriever in a RAG system?",
    "What is the role of the generator in a RAG system?",
    "What are common limitations of RAG systems?",
    "Why is chunking important in a RAG pipeline?",
]

# Ground truths written from the Wikipedia article BEFORE running the model
ground_truths = [
    "Retrieval-augmented generation (RAG) is an AI framework that enhances LLMs by retrieving relevant information from external knowledge sources before generating a response.",
    "RAG solves the problem of LLMs having outdated or incomplete parametric knowledge by allowing them to access up-to-date external information at inference time without retraining.",
    "RAG was introduced by Patrick Lewis and colleagues at Facebook AI Research (Meta) in their 2020 paper 'Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks'.",
    "A RAG pipeline consists of a document loader, a text splitter, an embedding model, a vector store for indexing, a retriever, a prompt template, and a language model generator.",
    "Unlike fine-tuning, which updates model weights and requires costly retraining, RAG retrieves external knowledge at inference time and does not modify model parameters.",
    "RAG can retrieve from vector databases, document stores, Wikipedia, PDFs, text files, web pages, and other structured or unstructured knowledge bases.",
    "The retriever finds the most semantically relevant document chunks from the knowledge base given the user query, typically using embedding-based similarity search.",
    "The generator (LLM) produces a grounded, coherent answer by combining the user's question with the retrieved context passages supplied in the prompt.",
    "Common limitations of RAG include retrieval failures when key information is absent, added latency from the retrieval step, context window constraints, and models that may ignore retrieved context.",
    "Chunking splits documents into short focused passages so each embedding represents a specific idea, enabling precise retrieval; overlapping chunks prevent important information from being cut off at boundaries.",
]


# Part B: Collect pipeline outputs
generated_answers  = []
retrieved_contexts = []

for i, q in enumerate(questions, 1):
    docs = retriever.invoke(q)
    context_texts = [doc.page_content for doc in docs]
    answer = rag_chain.invoke(q)
    generated_answers.append(answer)
    retrieved_contexts.append(context_texts)
    print(f"  [{i}/{len(questions)}] {q[:60]}...")


# Part C: Build the ragas 0.2.x EvaluationDataset
eval_dataset = EvaluationDataset(samples=[
    SingleTurnSample(
        user_input=q,
        response=a,
        retrieved_contexts=ctx,
        reference=gt,
    )
    for q, a, ctx, gt in zip(questions, generated_answers, retrieved_contexts, ground_truths)
])


# ── SELF CHECK ───────────────────────────────────────────────
assert len(questions)    >= 10, f"✗ Need 10+ questions, you have {len(questions)}"
assert len(questions)    == len(ground_truths), \
    "✗ questions and ground_truths must be the same length"
assert len(generated_answers)  == len(questions), \
    "✗ Run the collection loop first to fill generated_answers"
assert len(retrieved_contexts) == len(questions), \
    "✗ Run the collection loop first to fill retrieved_contexts"
assert isinstance(eval_dataset, EvaluationDataset), \
    "✗ Build eval_dataset as EvaluationDataset(samples=[...])"
assert len(eval_dataset) == len(questions), \
    f"✗ Expected {len(questions)} samples in eval_dataset"

print(f"✓ Golden dataset ready: {len(eval_dataset)} samples")
s0 = eval_dataset[0]
print(f"\nPreview — Row 1:")
print(f"  Q  : {s0.user_input}")
print(f"  GT : {s0.reference[:100]}")
print(f"  A  : {s0.response[:100]}...")
print(f"  Ctx: {len(s0.retrieved_contexts)} chunks retrieved")


---

## 📌 Git Checkpoint 4 — Golden Dataset Written

**What to note:** Ground truths are original intellectual work.
Committing them separately proves you wrote them before seeing the model's output.


In [ ]:
# ════════════════════════════════════════════════════════════════
# GIT CHECKPOINT 4
# ════════════════════════════════════════════════════════════════

NOTES = f"""
Exercises: 7 (golden evaluation dataset)
Status: ✓ / ⚠ / ✗   ← delete two

Questions written : {len(questions) if 'questions' in dir() else '???'}
Ground truth source: Wikipedia article (written before running the model)

Topics covered:
  -
  -
  -

Notes (what made writing ground truths harder than expected):
-
"""

import subprocess
staged = subprocess.run("git diff --cached --name-only",
                        shell=True, capture_output=True, text=True).stdout
assert ".env" not in staged, "🚨  .env is staged! Run: git reset HEAD .env"
print("✓ .env safety check passed")

subprocess.run("git add rag_eval_workflow.ipynb", shell=True)
q_count = len(questions) if 'questions' in dir() else '?'
subprocess.run(["git", "commit", "-m",
    f"[RAG-4] Golden dataset — {q_count} handwritten Q&A pairs\n\n{NOTES.strip()}"])
print("✓ Committed!  Push when ready: git push origin main")


---

## Exercise 8 — Run RAGAS Evaluation

### Concept: The 4 RAGAS Metrics

RAGAS uses the LLM itself as a judge to score your pipeline on 4 dimensions:

| Metric | Question it answers | Failure = |
|--------|--------------------|----|
| `faithfulness` | Is every claim in the answer supported by the retrieved context? | Hallucination |
| `answer_relevancy` | Does the answer actually address what was asked? | Vague / off-topic |
| `context_recall` | Does the retrieved context contain the ground truth information? | Wrong chunks retrieved |
| `context_precision` | Are most retrieved chunks relevant (low noise)? | Too much junk retrieved |

**Score scale: 0.0 (worst) → 1.0 (best)**

A strong production RAG system typically scores > 0.80 across all metrics.

### Running evaluation

```python
results = evaluate(
    dataset=eval_dataset,
    metrics=[faithfulness, answer_relevancy, context_recall, context_precision],
)
results_df = results.to_pandas()
```

> ⚠️ RAGAS makes LLM API calls internally to score each row.  
> With 10 questions × 4 metrics this takes **1–3 minutes** and costs ~**$0.05**.

**Instructions:**
1. Create a list `metrics` containing all 4 metric objects
2. Call `evaluate(dataset=eval_dataset, metrics=metrics)` — store as `results`
3. Convert to a DataFrame: `results_df = results.to_pandas()`
4. Display `results_df`

**Expected output:** A table with columns `question`, `faithfulness`, `answer_relevancy`, `context_recall`, `context_precision`.


In [ ]:
# ════════════════════════════════════════════════════════════════
# EXERCISE 8 — Run RAGAS Evaluation
# ════════════════════════════════════════════════════════════════

METRIC_COLS = ["faithfulness", "answer_relevancy", "context_recall", "context_precision"]

# Step 1: Instantiate the 4 metric objects (ragas 0.2.x uses classes)
metrics = [Faithfulness(), AnswerRelevancy(), ContextRecall(), ContextPrecision()]

# Step 2: Run evaluation — RAGAS calls the LLM internally as a judge
print("Running RAGAS evaluation — takes 1–3 minutes...")
results = evaluate(dataset=eval_dataset, metrics=metrics)

# Step 3: Convert to a pandas DataFrame
results_df = results.to_pandas()

# Step 4: Display
q_col = "user_input" if "user_input" in results_df.columns else "question"
display(results_df[[q_col] + METRIC_COLS])


# ── SELF CHECK ───────────────────────────────────────────────
assert metrics is not None and len(metrics) == 4, \
    "✗ Create a list of exactly 4 RAGAS metric objects"
assert results    is not None, "✗ Call evaluate() and store the result"
assert results_df is not None, "✗ Call results.to_pandas() and store as results_df"
assert all(col in results_df.columns for col in METRIC_COLS), \
    f"✗ DataFrame missing metric columns. Got: {list(results_df.columns)}"
print("\n✓ Evaluation complete!")
print(f"  Rows: {len(results_df)}  |  Columns: {list(results_df.columns)}")


---

## Exercise 9 — Analyze & Visualize Results

### Concept: Reading Your Scores

Print the aggregate (mean) score for each metric, then visualize per-question
scores to find patterns in failures.

**Score interpretation guide:**

| Score range | Meaning | Fix |
|------------|---------|-----|
| `faithfulness` < 0.7 | LLM is hallucinating — ignoring context | Tighten system prompt, lower temperature |
| `answer_relevancy` < 0.7 | Answers are vague or off-topic | Improve prompt clarity |
| `context_recall` < 0.7 | Retriever misses key chunks | Reduce chunk size, increase `k` |
| `context_precision` < 0.7 | Too much irrelevant context retrieved | Increase threshold, use MMR retrieval |

### Visualizations to build

1. **Bar chart** — one bar per metric showing the mean score (axes[0])
2. **Heatmap** — rows = questions (Q1–Q10), columns = 4 metrics, color = score (axes[1])

**Instructions:**
1. Print a formatted table of mean scores for all 4 metrics
2. Build the bar chart on `axes[0]` — add a dashed "target" line at 0.8
3. Build the heatmap on `axes[1]` with `seaborn.heatmap` using `cmap="RdYlGn"`
4. Identify the 3 worst questions using `nsmallest(3, "avg_score")`
5. Export `results_df` to `./rag_eval_results.csv`

**Expected output:** Score table, two-panel chart, worst-question list, CSV saved.


In [ ]:
# ════════════════════════════════════════════════════════════════
# EXERCISE 9 — Analyze & Visualize Results
# ════════════════════════════════════════════════════════════════

import matplotlib.pyplot as plt
import seaborn as sns

# Part A: Print aggregate scores
print("Mean RAGAS Scores:")
for metric in METRIC_COLS:
    score = results_df[metric].mean()
    bar   = "█" * int(score * 20) + "░" * (20 - int(score * 20))
    print(f"  {metric:<25} {score:.3f}  [{bar}]")


# Part B: Two-panel chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# axes[0]: Bar chart of mean scores
means = [results_df[m].mean() for m in METRIC_COLS]
colors = ["#4CAF50" if v >= 0.8 else "#FF9800" if v >= 0.6 else "#F44336" for v in means]
axes[0].bar(METRIC_COLS, means, color=colors)
axes[0].axhline(0.8, linestyle="--", color="gray", label="Target (0.8)")
axes[0].set_ylim(0, 1.05)
axes[0].set_title("Mean RAGAS Scores")
axes[0].set_ylabel("Score")
axes[0].tick_params(axis="x", rotation=15)
for i, v in enumerate(means):
    axes[0].text(i, v + 0.02, f"{v:.2f}", ha="center", fontsize=9)
axes[0].legend()

# axes[1]: Heatmap — per-question scores
heatmap_data = results_df[METRIC_COLS].copy()
heatmap_data.index = [f"Q{i+1}" for i in range(len(heatmap_data))]
sns.heatmap(heatmap_data, ax=axes[1], annot=True, fmt=".2f",
            cmap="RdYlGn", vmin=0, vmax=1, linewidths=0.5)
axes[1].set_title("Per-Question Scores")

plt.tight_layout()
plt.show()


# Part C: Worst questions
results_df["avg_score"] = results_df[METRIC_COLS].mean(axis=1)
worst = results_df.nsmallest(3, "avg_score")
q_col = "user_input" if "user_input" in results_df.columns else "question"
print("\n3 Worst-Scoring Questions:")
for _, row in worst.iterrows():
    scores_str = "  ".join(f"{m}={row[m]:.2f}" for m in METRIC_COLS)
    print(f"  Q: {row[q_col]}")
    print(f"     avg={row['avg_score']:.3f}  {scores_str}")


# Part D: Export
results_df.to_csv("./rag_eval_results.csv", index=False)
print("\n✓ Results saved to rag_eval_results.csv")


# ── SELF CHECK ───────────────────────────────────────────────
import os
assert os.path.exists("./rag_eval_results.csv"), \
    "✗ CSV not found — did you call results_df.to_csv('./rag_eval_results.csv', index=False)?"
assert "avg_score" in results_df.columns, \
    "✗ avg_score column missing — compute it with results_df[METRIC_COLS].mean(axis=1)"
print("✓ CSV saved.  Run `make check-eval` to validate scores.")


---

## 📌 Git Checkpoint 5 — Project Complete

**This is your final commit.** It auto-fills your RAGAS scores into the commit
message so your portfolio commit history shows the actual numbers.


In [ ]:
# ════════════════════════════════════════════════════════════════
# GIT CHECKPOINT 5 — FINAL
# ════════════════════════════════════════════════════════════════

# Auto-fill scores from results_df
try:
    scores_str = "\n".join(
        f"  {m:<22}: {results_df[m].mean():.3f}" for m in METRIC_COLS
    )
    weakest = min(METRIC_COLS, key=lambda m: results_df[m].mean())
except Exception:
    scores_str = "  (run Exercise 8 first)"
    weakest    = "unknown"

NOTES = f"""
Exercises: 8 (RAGAS evaluation), 9 (analysis + visualization)
Status: ✓ / ⚠ / ✗   ← delete two

RAGAS Scores (mean across {len(questions) if 'questions' in dir() else '?'} questions):
{scores_str}

Weakest metric  : {weakest}
Root cause      : ???   ← explain why this metric is low
Planned fix     : ???   ← what would you change first

Key takeaways:
-
-
"""

import subprocess
staged = subprocess.run("git diff --cached --name-only",
                        shell=True, capture_output=True, text=True).stdout
assert ".env" not in staged, "🚨  .env is staged! Run: git reset HEAD .env"
print("✓ .env safety check passed")

subprocess.run("git add rag_eval_workflow.ipynb rag_eval_results.csv 2>/dev/null || true",
               shell=True)
subprocess.run(["git", "commit", "-m",
    f"[RAG-5] FINAL — evaluation complete\n\n{NOTES.strip()}"])

# Push everything
print("\nPushing all checkpoints to GitHub...")
result = subprocess.run("git push origin main",
                        shell=True, capture_output=True, text=True)
print(result.stdout or result.stderr)
print("\n" + "═"*50)
print("  PROJECT COMPLETE")
print("  https://github.com/Dashakid/RAG-101")
print("  Run: git log --oneline  to see your history")
print("═"*50)


---

## 🎓 You're Done

### What You Built

```
Wikipedia article  ──►  WebBaseLoader  ──►  RecursiveCharacterTextSplitter
                                                        │
                              OpenAIEmbeddings  +  ChromaDB
                                                        │
User Question  ──────────────────────────►  Retriever (k=4)
                                                        │
                        ChatPromptTemplate  +  gpt-4o-mini
                                                        │
                                    Answer  ──►  RAGAS  ──►  CSV
```

---

### Reflection Questions

Write your answers in a new markdown cell below each one:

1. **What was your lowest-scoring metric?**  
   What does that tell you about your pipeline's specific weakness?

2. **Look at your 3 worst questions.**  
   Is there a pattern — all from the same section, all a certain type?

3. **What would you change first** to improve your lowest score?  
   (chunk size / k / prompt / LLM choice)

---

### Next Challenges

| Challenge | What to change |
|-----------|---------------|
| Improve context recall | Increase `k` from 4 to 6 and re-evaluate |
| Reduce retrieval noise | Switch to `search_type="mmr"` |
| Try different documents | Load a PDF with `PyPDFLoader` |
| Tune chunk size | Try 256 and 1024 — compare scores |
| Add memory | Wrap chain with `ConversationBufferMemory` |
| Deploy it | Wrap `rag_chain.invoke()` in a FastAPI route |
